# DE-3) library `csv` สำหรับงาน Data Engineer เบื้องต้น

ไฟล์ CSV (Comma-Separated Values) คือรูปแบบไฟล์ข้อมูลที่ใช้บ่อยที่สุดในงาน Data Engineering

ตัวอย่างงานจริง:
- รับไฟล์ข้อมูลรายวันจากระบบหนึ่ง
- รวมหลายไฟล์ CSV เข้าด้วยกัน
- แปลงข้อมูลก่อนส่งต่อไปฐานข้อมูลหรือ Data Warehouse

ในบทนี้ เราจะเรียนรู้การอ่านและเขียนไฟล์ CSV ด้วย **library `csv` (มาตรฐานของ Python)**

---

## สิ่งที่คุณจะได้จากบทนี้
- เข้าใจโครงสร้างไฟล์ CSV
- อ่านไฟล์ CSV ด้วย `csv.reader`
- อ่านไฟล์แบบมี header ด้วย `csv.DictReader`
- เขียนไฟล์ CSV ด้วย `csv.writer`
- ทำ workflow เบื้องต้น: อ่าน → แปลง → เขียนไฟล์ใหม่

---

## ภาพรวมเนื้อหา
1. โครงสร้างไฟล์ CSV
2. อ่านไฟล์ด้วย `csv.reader`
3. อ่านไฟล์แบบมี header ด้วย `DictReader`
4. เขียนไฟล์ด้วย `csv.writer`
5. ตัวอย่าง mini pipeline

## 1) โครงสร้างไฟล์ CSV

ตัวอย่าง CSV:

```
name,age,score
Alice,20,80
Bob,22,75
```

- บรรทัดแรก = header (ชื่อคอลัมน์)
- บรรทัดถัดไป = ข้อมูลแต่ละแถว

Data Engineer ต้องเข้าใจว่า CSV จริงอาจมี:
- ตัวคั่นไม่ใช่ comma (เช่น `;`)
- encoding ต่างกัน
- คอลัมน์ไม่ครบ

## 2) สร้างไฟล์ CSV ตัวอย่าง

In [2]:
import os
import csv

os.makedirs("data_csv", exist_ok=True)

sample_path = os.path.join("data_csv", "sample.csv")

with open(sample_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["name", "age", "score"])
    writer.writerow(["Alice", 20, 80])
    writer.writerow(["Bob", 22, 75])

sample_path

'data_csv/sample.csv'

## 3) อ่านไฟล์ด้วย `csv.reader`

In [3]:
with open(sample_path, "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)

['name', 'age', 'score']
['Alice', '20', '80']
['Bob', '22', '75']


สังเกตว่า:
- ข้อมูลแต่ละแถวจะได้เป็น list
- ตัวเลขยังเป็น string ต้องแปลงเองถ้าจะคำนวณ

## 4) อ่านไฟล์แบบมี header ด้วย `csv.DictReader`

In [4]:
with open(sample_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(row)

{'name': 'Alice', 'age': '20', 'score': '80'}
{'name': 'Bob', 'age': '22', 'score': '75'}


ข้อดีของ DictReader:
- เข้าถึงข้อมูลด้วยชื่อคอลัมน์ เช่น row["score"]
- อ่านง่ายกว่าในงานจริง

## 5) ตัวอย่างแปลงข้อมูลระหว่างอ่านไฟล์

In [5]:
with open(sample_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        score = int(row["score"])
        if score >= 80:
            print(row["name"], "-> High score")

Alice -> High score


## 6) เขียนไฟล์ CSV ใหม่ด้วย `csv.writer`

In [6]:
output_path = os.path.join("data_csv", "output.csv")

with open(sample_path, "r", encoding="utf-8") as f_in,      open(output_path, "w", newline="", encoding="utf-8") as f_out:
    
    reader = csv.DictReader(f_in)
    writer = csv.writer(f_out)
    
    writer.writerow(["name", "score", "status"])
    
    for row in reader:
        score = int(row["score"])
        status = "Pass" if score >= 80 else "Normal"
        writer.writerow([row["name"], score, status])

output_path

'data_csv/output.csv'

ตรวจสอบไฟล์ใหม่

In [7]:
with open(output_path, "r", encoding="utf-8") as f:
    print(f.read())

name,score,status
Alice,80,Pass
Bob,75,Normal



## 7) Mini Pipeline ตัวอย่าง

Workflow แบบ Data Engineer:

1) อ่านไฟล์ CSV
2) แปลง/คัดกรองข้อมูล
3) เขียนไฟล์ใหม่
4) ส่งต่อให้ขั้นตอนถัดไป

คุณเพิ่งทำ pipeline แรกของคุณแล้ว

## 8) แบบฝึกหัด

ให้ทำดังนี้:

1) สร้างไฟล์ `products.csv` มีคอลัมน์:
   - product
   - price

2) อ่านไฟล์นั้น
3) ถ้า price > 1000 ให้เพิ่มคอลัมน์ใหม่ชื่อ `expensive` เป็น True
4) เขียนไฟล์ใหม่ชื่อ `products_output.csv`

## 🔎 อธิบายพารามิเตอร์สำคัญของการเปิดและเขียนไฟล์ CSV

แม้ว่าเราจะใช้ค่า default ได้ในหลายกรณี  
แต่สำหรับสาย Data Engineer การเข้าใจพารามิเตอร์เหล่านี้ถือว่า “สำคัญมาก”

---

### 1️⃣ `open(file, mode, encoding, newline)`

ตัวอย่างที่เราใช้:

```python
open(sample_path, "w", newline="", encoding="utf-8")
```

#### ▪ `mode`
กำหนดโหมดการเปิดไฟล์

- `"r"` → อ่านไฟล์
- `"w"` → เขียนไฟล์ (ถ้ามีไฟล์เดิมจะถูกเขียนทับ)
- `"a"` → เขียนต่อท้าย (append)
- `"rb"` / `"wb"` → โหมด binary

---

#### ▪ `encoding`
กำหนดรูปแบบการเข้ารหัสตัวอักษร

ค่าที่พบบ่อย:
- `"utf-8"` (แนะนำและเป็นมาตรฐาน)
- `"utf-8-sig"` (บางไฟล์ Excel ใช้)
- `"cp874"` (ภาษาไทยใน Windows เก่า)
- `"latin-1"` (ไฟล์ยุโรปบางประเภท)

⚠️ ถ้า encoding ผิด จะเจอ error เช่น:
```
UnicodeDecodeError
```

Data Engineer ต้องตรวจสอบ encoding ให้ถูกต้องก่อนอ่านไฟล์

---

#### ▪ `newline=""` (สำคัญมากตอนเขียน CSV)

เวลาสร้างไฟล์ CSV บน Windows  
ถ้าไม่ใส่ `newline=""`  
จะเกิดบรรทัดว่างคั่นทุกแถว

ดังนั้น best practice คือ:

```python
open(file, "w", newline="", encoding="utf-8")
```

---

### 2️⃣ พารามิเตอร์ของ `csv.reader`

```python
csv.reader(file, delimiter=",", quotechar='"')
```

#### ▪ `delimiter`
ตัวคั่นคอลัมน์

ค่าที่พบบ่อย:
- `","` (มาตรฐาน CSV)
- `";"` (บางประเทศใช้)
- `"\t"` (ไฟล์ TSV)

ถ้า delimiter ไม่ถูกต้อง → คอลัมน์จะแตกผิดรูป

---

#### ▪ `quotechar`
ตัวครอบข้อความ เช่น:

```
"Bangkok, Thailand"
```

ช่วยให้ comma ภายในข้อความไม่ถูกตีความเป็นตัวแบ่งคอลัมน์

---

### 3️⃣ พารามิเตอร์ของ `csv.writer`

```python
csv.writer(file, delimiter=",", quoting=csv.QUOTE_MINIMAL)
```

#### ▪ `quoting`

- `csv.QUOTE_MINIMAL` (default)
- `csv.QUOTE_ALL`
- `csv.QUOTE_NONNUMERIC`
- `csv.QUOTE_NONE`

ในงานจริง บางระบบต้องการ format เฉพาะ  
Data Engineer ต้องเข้าใจการตั้งค่าเหล่านี้

---

## 🎯 สรุปสำคัญ

ถึงแม้เราจะใช้ค่า default ได้  
แต่ในงาน production:

- encoding ต้องตรวจสอบเสมอ
- delimiter ต้องเช็คจากไฟล์ต้นทาง
- newline สำคัญมากตอนเขียนไฟล์
- mode ต้องเลือกให้ถูกเพื่อไม่ให้ข้อมูลสูญหาย

ความเข้าใจเรื่อง configuration คือพื้นฐานสำคัญของ Data Engineer

## 9) สรุป (Checklist)

หลังจบบทนี้ คุณควรทำได้:
- ✅ อ่านไฟล์ CSV ด้วย `csv.reader`
- ✅ อ่านไฟล์แบบมี header ด้วย `csv.DictReader`
- ✅ เขียนไฟล์ CSV ใหม่ด้วย `csv.writer`
- ✅ แปลงข้อมูลระหว่างอ่านไฟล์
- ✅ เข้าใจ workflow pipeline เบื้องต้น

➡️ บทถัดไป: **pandas** เพื่อทำงานกับ CSV ได้สะดวกและมีประสิทธิภาพมากขึ้น